# Program Intake Triage Agent
### Enterprise program intake, triaged — with a human always in the loop

A multi-agent **ADK-Python + Gemini** pipeline that turns a raw program request into a
**structured, human-reviewed routing recommendation**. It never auto-approves or auto-rejects.

**Pipeline:** Intake Parser → Triage Scorer → Router / Capacity Check → guardrail

_Capstone — AI Agents Intensive · Track: Agents for Business · by Raj Sarangam_


## 1. Install dependencies


In [1]:
!pip -q install google-adk google-genai pydantic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.0/306.0 kB 16.6 MB/s eta 0:00:00


## 2. Clone the code from GitHub

Pulls the `intake_triage` package (the agent) from the public repo.


In [2]:
import os, sys
%cd /kaggle/working
!rm -rf PortfolioFlow-AI
!git clone https://github.com/RajSarangam/PortfolioFlow-AI.git
%cd /kaggle/working/PortfolioFlow-AI
REPO = '/kaggle/working/PortfolioFlow-AI'
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('files:', os.listdir())


/kaggle/working
Cloning into 'PortfolioFlow-AI'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 76 (delta 14), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 42.78 KiB | 7.13 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/kaggle/working/PortfolioFlow-AI
files: ['mcp_server', 'WRITEUP.md', 'intake_triage', 'README.md', 'notebooks', 'LICENSE', '.gitignore', '.git', 'requirements.txt', 'data', 'portfolio-ai.ipynb']


## 3. Load your Gemini API key from Kaggle Secrets

Add-ons → Secrets → add a secret named `GOOGLE_API_KEY` (your AI Studio key).


In [3]:
from kaggle_secrets import UserSecretsClient
os.environ['GOOGLE_API_KEY'] = UserSecretsClient().get_secret('GOOGLE_API_KEY')
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'FALSE'  # use AI Studio, not Vertex
print('Gemini key loaded:', bool(os.environ.get('GOOGLE_API_KEY')))


Gemini key loaded: True


## 4. Build the runner

`triage()` creates its own session, runs all three agents in order, and returns the
structured JSON each stage wrote to session state.


In [4]:
import warnings, json, re, itertools
warnings.filterwarnings('ignore')

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as gt
from intake_triage import root_agent

session_service = InMemorySessionService()
runner = Runner(agent=root_agent, app_name='intake_triage', session_service=session_service)
_counter = itertools.count(1)

def _safe_json(text):
    """Parse a JSON string that may be wrapped in code fences."""
    if not text:
        return None
    cleaned = re.sub(r'^```(json)?|```$', '', text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except Exception:
        return {'_raw': text}

def triage(request_text):
    """Run parse -> score -> route and return the final session state."""
    uid, sid = 'pm', f's-{next(_counter)}'   # unique session per call
    session_service.create_session_sync(app_name='intake_triage',
                                        user_id=uid, session_id=sid, state={})
    msg = gt.Content(role='user', parts=[gt.Part.from_text(text=request_text)])
    for _ in runner.run(user_id=uid, session_id=sid, new_message=msg):
        pass                                  # drive all three agents to completion
    s = session_service.get_session_sync(app_name='intake_triage',
                                         user_id=uid, session_id=sid)
    return s.state

print('runner ready')


runner ready


## 5. Run a sample request


In [5]:
SAMPLE = (
    'Hi team, Sales wants a single view of each customer pulling together Salesforce '
    'records, support tickets, and product usage so reps can see churn risk. We think '
    'this is important and would like it within the quarter. Not sure who owns it.'
)

state = triage(SAMPLE)
parsed     = _safe_json(state.get('parsed_request'))
assessment = _safe_json(state.get('triage_assessment'))
routing    = _safe_json(state.get('routing_recommendation'))

print('PARSED:');       print(json.dumps(parsed, indent=2))
print('\nASSESSMENT:'); print(json.dumps(assessment, indent=2))
print('\nROUTING:');    print(json.dumps(routing, indent=2))


Deprecated. Please migrate to the async method.


[guardrail] tool call: find_related_projects args={'keywords': 'Customer view Salesforce Support tickets Product usage Churn risk Sales efficiency'}
[guardrail] tool call: lookup_capacity args={'team_name': 'CRM & Revenue'}


Deprecated. Please migrate to the async method.


PARSED:
{
  "requestor": "Sales",
  "business_unit": "Sales",
  "problem_statement": "Sales wants a single view of each customer pulling together Salesforce records, support tickets, and product usage.",
  "desired_outcome": "Sales representatives can see churn risk.",
  "urgency": "high",
  "rough_size": null,
  "dependencies": [],
  "keywords": [
    "Customer view",
    "Salesforce",
    "Support tickets",
    "Product usage",
    "Churn risk",
    "Sales efficiency"
  ],
  "missing_info": [
    "rough_size",
    "owner"
  ]
}

ASSESSMENT:
{
  "scores": {
    "value": 5,
    "effort": 5,
    "risk": 5,
    "strategic_alignment": 3
  },
  "rationale": {
    "value": "Providing sales representatives with a single view of customer data and churn risk directly impacts revenue retention and sales efficiency, which are enterprise-level concerns.",
    "effort": "Integrating data from Salesforce, support tickets, and product usage into a single view, especially for churn prediction, is a c

## 6. Apply the human-in-the-loop guardrail

Assembles the final **recommendation** — with a confidence level and a de-duplicated list
of review flags. It is advisory only: a human makes every approve/reject decision.


In [6]:
from intake_triage.skills.triage_rubric import score_to_tier
from intake_triage.guardrails import screen_request, enforce_human_in_the_loop

# Deterministic priority tier (passed positionally so key names never clash).
if assessment and 'scores' in assessment:
    sc = assessment['scores']
    tier = score_to_tier(sc['value'], sc['effort'], sc['risk'], sc['strategic_alignment'])
else:
    tier = {}

# Collect review flags from every stage, then de-duplicate while preserving order.
raw_flags = []
raw_flags += screen_request(SAMPLE)['input_flags']
raw_flags += (assessment or {}).get('missing_info', [])
raw_flags += (routing or {}).get('open_questions', [])
seen = set()
flags = [f for f in raw_flags if not (f in seen or seen.add(f))]

final = enforce_human_in_the_loop(routing or {}, flags)
final['priority'] = tier

print(json.dumps(final, indent=2))


{
  "recommendation": "needs_more_info",
  "decision_authority": "human",
  "auto_action_taken": false,
  "human_review_required": true,
  "confidence": "low",
  "review_flags": [
    "rough_size",
    "owner",
    "strategic_alignment",
    "How does this request align with or differ from 'Customer Health Scoring' (PRJ-1009) and 'Unified Customer 360' (PRJ-1001)?",
    "What is the rough size estimate for this project in terms of effort and duration?",
    "Confirm the owner for this project."
  ],
  "router_detail": {
    "recommendation": "needs_more_info",
    "target_team": "CRM & Revenue",
    "capacity_state": "tight",
    "related_projects": [
      {
        "id": "PRJ-1009",
        "name": "Customer Health Scoring",
        "overlap_score": 5
      },
      {
        "id": "PRJ-1001",
        "name": "Unified Customer 360",
        "overlap_score": 3
      },
      {
        "id": "PRJ-1004",
        "name": "Self-Service Support Portal",
        "overlap_score": 2
      },


## Concepts demonstrated

| Concept | Where |
|---|---|
| Multi-agent system (ADK) | orchestrator over Parser / Scorer / Router |
| Sessions & state management | each agent's `output_key` -> session state, read by the next |
| Agent skills | codified scoring rubric in `skills/triage_rubric.py` |
| Security / guardrail layer | input screening, tool-call guard, human-in-the-loop enforcement |
| MCP server integration *(MCP-ready)* | `mcp_server/portfolio_mcp_server.py`, swap documented in its header |

_Advisory by design — a human reviewer makes every approve/reject decision._
